# 🙃 Downside-Up Complaint Bureau
### NormalObjects — Creative Complaint Handler (LangChain Lab)

This notebook builds **Becma's Chaos Mode**: a LangChain agent that handles complaints about the Normal Objects universe using flexible, creative tool-calling.

**Steps:**
1. Setup & install
2. Define creative tools
3. Build the agent
4. Test with sample complaints
5. Analyse tool usage

## Step 1 — Setup & Install

In [1]:
# Install required packages
!pip install -q langchain langchain-openai python-dotenv

In [3]:
import os, random, json
from typing import List

# LangChain 0.3+ import paths
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage

import langchain
print('✅ Imports OK  |  LangChain version:', langchain.__version__)

✅ Imports OK  |  LangChain version: 1.3.4


In [4]:
# Set your OpenAI API key
# load from a .env file (recommended for local dev)
# from dotenv import load_dotenv
# load_dotenv()

# Initialise the LLM
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.7)
print('✅ LLM ready:', llm.model_name)

✅ LLM ready: gpt-4o-mini


## Step 2 — Create Creative Tools

Four themed tools the agent can call in any order it chooses.

In [5]:
@tool
def consult_demogorgon(complaint: str) -> str:
    """Get the Demogorgon's perspective on a complaint about the Upside Down.

    The Demogorgon is a creature from the Upside Down. It might have insights
    about interdimensional inconsistencies, but its perspective is... unique.

    Args:
        complaint: The complaint about the Upside Down

    Returns:
        The Demogorgon's perspective (creative and possibly chaotic)
    """
    responses = [
        f"The Demogorgon tilts its head. It seems confused by '{complaint}'. "
        f"Perhaps the issue is that you're thinking in three dimensions?",
        f"The Demogorgon makes a sound that might be agreement. It suggests "
        f"the problem might be temporal — things work differently in the Upside Down's time.",
        f"The Demogorgon appears to be eating something. It doesn't understand "
        f"'{complaint}' — maybe consistency isn't a priority there?"
    ]
    return random.choice(responses)


@tool
def check_hawkins_records(query: str) -> str:
    """Search Hawkins historical records for information.

    Hawkins has a long history of strange occurrences. These records might
    contain clues about patterns or explanations.

    Args:
        query: What to search for in the records

    Returns:
        Information from Hawkins historical records
    """
    records = {
        'portal': 'Records show portals have opened on various dates with no clear pattern. '
                  'Weather, electromagnetic activity, and unknown factors seem involved.',
        'monsters': 'Historical records indicate creatures from the Upside Down behave '
                    'differently based on environmental factors, time of day, and proximity '
                    'to certain individuals.',
        'psychics': 'Records show that psychic abilities vary greatly. Some individuals can '
                    'move objects but not see the future; others can see visions but not move things.',
        'electricity': 'Hawkins has a history of electrical anomalies. Records suggest a connection '
                       'between the Upside Down and electromagnetic fields.'
    }
    for key, value in records.items():
        if key in query.lower():
            return value
    return (f"Records don't contain specific information about '{query}', but they note "
            f"that many unexplained events have occurred in Hawkins over the years.")


@tool
def cast_interdimensional_spell(problem: str, creativity_level: str = 'medium') -> str:
    """Suggest a creative interdimensional spell to fix a problem.

    Sometimes the best solution is a creative one that doesn't follow normal rules.
    This tool suggests imaginative fixes for Upside Down problems.

    Args:
        problem: The problem to solve
        creativity_level: How creative to be — 'low', 'medium', or 'high'

    Returns:
        A creative spell or solution suggestion
    """
    creativity_multiplier = {'low': 1, 'medium': 2, 'high': 3}.get(creativity_level, 2)
    spells = [
        f"Try chanting 'Bemca Becma Becma' three times while holding a Walkman. "
        f"This might recalibrate the interdimensional frequencies related to: {problem}",
        f"Create a salt circle and place a compass in the centre. "
        f"The magnetic anomalies might help stabilise: {problem}",
        f"Play 'Running Up That Hill' backwards at the exact location of the issue. "
        f"The temporal resonance could fix: {problem}",
        f"Gather three items: a lighter, a compass, and something personal. "
        f"Arrange them in a triangle while focusing on: {problem}."
    ]
    selected = random.sample(spells, min(creativity_multiplier, len(spells)))
    return '\n'.join(selected)


@tool
def gather_party_wisdom(question: str) -> str:
    """Ask the D&D party (Mike, Dustin, Lucas, Will) for their collective wisdom.

    The party has solved many mysteries together. Their combined knowledge
    and different perspectives can provide useful insights.

    Args:
        question: The question or problem to put to the party

    Returns:
        The party's collective wisdom and suggestions
    """
    party_responses = {
        'portal': "Mike: 'Portals are unpredictable but usually open near strong emotional events "
                  "or electromagnetic disturbances.' Dustin: 'Also, they seem to follow some kind "
                  "of pattern related to the Mind Flayer's activity.'",
        'monsters': "Lucas: 'Demogorgons are territorial but also opportunistic.' "
                    "Will: 'They can sense fear and strong emotions — maybe that's why they "
                    "act differently sometimes.'",
        'psychics': "Mike: 'El's powers seem connected to her emotional state.' "
                    "Dustin: 'And they're limited by her physical and mental energy. "
                    "That's probably why she can't do everything.'",
        'electricity': "Lucas: 'The Upside Down seems to interfere with electrical systems.' "
                       "Dustin: 'But it also creates strange connections — like a feedback loop.'"
    }
    for key, response in party_responses.items():
        if key in question.lower():
            return response
    return ("The party huddles together. Mike: 'This is a tough one.' "
            "Dustin: 'We need more information.' Lucas: 'Let's think about what we know.' "
            "Will: 'Maybe we should consult other sources?'")


# Collect all tools
tools = [consult_demogorgon, check_hawkins_records, cast_interdimensional_spell, gather_party_wisdom]

print(f'✅ Created {len(tools)} creative tools:')
for t in tools:
    print(f'  • {t.name}: {t.description[:70]}...')

✅ Created 4 creative tools:
  • consult_demogorgon: Get the Demogorgon's perspective on a complaint about the Upside Down....
  • check_hawkins_records: Search Hawkins historical records for information.

Hawkins has a long...
  • cast_interdimensional_spell: Suggest a creative interdimensional spell to fix a problem.

Sometimes...
  • gather_party_wisdom: Ask the D&D party (Mike, Dustin, Lucas, Will) for their collective wis...


## Step 3 — Build the Agent

A prompt that encourages creative, freeform problem-solving, then wired up to an `AgentExecutor`.

In [7]:
SYSTEM_PROMPT = """You are Becma, head of the Downside-Up Complaint Bureau.
Handle complaints about the Normal Objects universe creatively.

You have four tools — use them in any order you judge best:
- consult_demogorgon   : creature perspective on Upside Down issues
- check_hawkins_records: historical patterns and documented evidence
- cast_interdimensional_spell: imaginative magical fixes
- gather_party_wisdom  : grounded insights from the D&D party

Guidelines:
- Be entertaining and creative
- Combine multiple tool results into one coherent answer
- Embrace the chaos — the universe is inconsistent by design
- End every response with an actionable (if absurd) recommendation
"""

# Bind tools to the LLM — this is the modern replacement for create_openai_tools_agent
llm_with_tools = llm.bind_tools(tools)

# Build a lookup so we can execute whichever tool the LLM calls by name
tool_map = {t.name: t for t in tools}


def run_agent(complaint: str, max_iterations: int = 6, verbose: bool = True) -> dict:
    """
    LCEL agent loop — replaces AgentExecutor.
    Keeps calling tools until the LLM produces a final text response.
    Returns {'output': str, 'intermediate_steps': list}
    """
    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=complaint),
    ]
    intermediate_steps = []   # (tool_name, tool_output) pairs — mirrors AgentExecutor format

    for iteration in range(max_iterations):
        response = llm_with_tools.invoke(messages)
        messages.append(response)  # add AI message to history

        # If no tool calls, the LLM is done — return the text
        if not response.tool_calls:
            return {'output': response.content, 'intermediate_steps': intermediate_steps}

        # Execute every tool the LLM requested
        for tc in response.tool_calls:
            tool_name = tc['name']
            tool_args = tc['args']
            tool_id   = tc['id']

            if verbose:
                print(f'  🔧 Calling {tool_name}({tool_args})')

            if tool_name in tool_map:
                tool_result = tool_map[tool_name].invoke(tool_args)
            else:
                tool_result = f'Unknown tool: {tool_name}'

            if verbose:
                print(f'     ↳ {str(tool_result)[:120]}')

            intermediate_steps.append((tool_name, tool_result))
            messages.append(ToolMessage(content=str(tool_result), tool_call_id=tool_id))

    # Safety fallback if we hit max_iterations
    return {'output': 'Max iterations reached without a final answer.', 'intermediate_steps': intermediate_steps}


print('✅ Agent loop ready (LCEL style)')

✅ Agent loop ready (LCEL style)


## Step 4 — Test with Sample Complaints

In [8]:
complaints = [
    "Why do demogorgons sometimes eat people and sometimes don't?",
    "The portal opens on different days — is there a schedule?",
    "Why can some psychics see the Upside Down and others can't?",
    "Why do creatures and power lines react so strangely together?",
]

all_results = []

def handle_complaint(complaint: str) -> dict:
    print(f"\n{'='*65}")
    print(f'📬 COMPLAINT: {complaint}')
    print(f"{'='*65}")
    result = run_agent(complaint, verbose=True)
    print(f"\n💬 FINAL RESPONSE:\n{result['output']}")
    return result

# Run first 3 complaints (add more if you like)
for c in complaints[:3]:
    result = handle_complaint(c)
    all_results.append({'complaint': c, 'result': result})


📬 COMPLAINT: Why do demogorgons sometimes eat people and sometimes don't?
  🔧 Calling consult_demogorgon({'complaint': "Why do demogorgons sometimes eat people and sometimes don't?"})
     ↳ The Demogorgon makes a sound that might be agreement. It suggests the problem might be temporal — things work differentl
  🔧 Calling check_hawkins_records({'query': 'Demogorgon eating habits'})
     ↳ Records don't contain specific information about 'Demogorgon eating habits', but they note that many unexplained events 
  🔧 Calling gather_party_wisdom({'question': "Why do demogorgons sometimes eat people and sometimes don't?"})
     ↳ The party huddles together. Mike: 'This is a tough one.' Dustin: 'We need more information.' Lucas: 'Let's think about w

💬 FINAL RESPONSE:
Ah, the enigmatic eating habits of Demogorgons! A mystery that has puzzled many, but fear not, for we shall unravel it together with a blend of perspectives.

First, from the Demogorgon's perspective: they seem to assert that the

In [13]:
complaints = [
    "Why do demogorgons look like aliens?",
    "Why does the Upside Down exist?",
    "Why do Vecna's followers eat kids?",
    "Why does Will have a scar?",
]

all_results = []

def handle_complaint(complaint: str) -> dict:
    print(f"\n{'='*65}")
    print(f'📬 COMPLAINT: {complaint}')
    print(f"{'='*65}")
    result = run_agent(complaint, verbose=True)
    print(f"\n💬 FINAL RESPONSE:\n{result['output']}")
    return result

# Run first 4 complaints
for c in complaints[:4]:
    result = handle_complaint(c)
    all_results.append({'complaint': c, 'result': result})


📬 COMPLAINT: Why do demogorgons look like aliens?
  🔧 Calling consult_demogorgon({'complaint': 'Why do demogorgons look like aliens?'})
     ↳ The Demogorgon tilts its head. It seems confused by 'Why do demogorgons look like aliens?'. Perhaps the issue is that yo
  🔧 Calling check_hawkins_records({'query': 'Demogorgon appearance and alien resemblance'})
     ↳ Records don't contain specific information about 'Demogorgon appearance and alien resemblance', but they note that many 
  🔧 Calling gather_party_wisdom({'question': 'Why do demogorgons look like aliens?'})
     ↳ The party huddles together. Mike: 'This is a tough one.' Dustin: 'We need more information.' Lucas: 'Let's think about w
  🔧 Calling cast_interdimensional_spell({'problem': 'Explain why demogorgons look like aliens', 'creativity_level': 'high'})
     ↳ Play 'Running Up That Hill' backwards at the exact location of the issue. The temporal resonance could fix: Explain why 

💬 FINAL RESPONSE:
Ah, the age-old question of w

## Step 5 — Analyse Tool Usage

Look at which tools were called, in what order, and how often.

In [14]:
class ToolUsageTracker:
    def __init__(self, tool_list):
        self.counts = {t.name: 0 for t in tool_list}
        self.sequences = []

    def ingest(self, result: dict, complaint: str):
        seq = []
        for tool_name, _ in result.get('intermediate_steps', []):
            self.counts[tool_name] = self.counts.get(tool_name, 0) + 1
            seq.append(tool_name)
        self.sequences.append({'complaint': complaint, 'sequence': seq})

    def report(self):
        total = sum(self.counts.values())
        most = max(self.counts, key=self.counts.get) if total else 'N/A'
        print('\n' + '='*65)
        print('📊  TOOL USAGE ANALYSIS')
        print('='*65)
        print(f'Total tool calls : {total}')
        print(f'Most-used tool   : {most}\n')
        print('Counts:')
        for name, count in sorted(self.counts.items(), key=lambda x: -x[1]):
            print(f'  {name:<38} {"█" * count} ({count})')
        print('\nSequences per complaint:')
        for item in self.sequences:
            seq = ' → '.join(item['sequence']) or '(no tools)'
            print(f"  • {item['complaint'][:48]:<48}  {seq}")

tracker = ToolUsageTracker(tools)
for item in all_results:
    tracker.ingest(item['result'], item['complaint'])
tracker.report()


📊  TOOL USAGE ANALYSIS
Total tool calls : 13
Most-used tool   : consult_demogorgon

Counts:
  consult_demogorgon                     ████ (4)
  check_hawkins_records                  ████ (4)
  gather_party_wisdom                    ████ (4)
  cast_interdimensional_spell            █ (1)

Sequences per complaint:
  • Why do demogorgons look like aliens?              consult_demogorgon → check_hawkins_records → gather_party_wisdom → cast_interdimensional_spell
  • Why does the Upside Down exist?                   check_hawkins_records → consult_demogorgon → gather_party_wisdom
  • Why do Vecna's followers eat kids?                consult_demogorgon → check_hawkins_records → gather_party_wisdom
  • Why does Will have a scar?                        check_hawkins_records → consult_demogorgon → gather_party_wisdom


## Reflection — notes for `lab_summary.md`

Answer these in your one-paragraph summary:
- Which tools did the agent pick most often, and did the order surprise you?
- What would break if the order were forced/fixed (as LangGraph will do in Lab 2)?
- When would you choose a **freeform LCEL agent** vs a **strict LangGraph workflow**?

**Rule of thumb:** freeform = open-ended / exploratory tasks; LangGraph = ordered pipelines where auditability and determinism matter.

## Optional Extensions

- **Add Eleven's tool**: `consult_eleven(vision: str)` — psychic insights with emotional weight
- **Dynamic tool responses**: replace the hard-coded strings with a second `llm.invoke()` call inside each tool
- **Conversation memory**: add `ConversationBufferMemory` so the agent references earlier complaints
- **Streamlit UI**: wrap `handle_complaint` in a simple web form